# CycPeptMPDB-4D + Assay 통합 데이터셋 분석

**대상 파일**: `data/CycPeptMPDB-4D_with_assay_descriptors.csv` (5160 펩타이드 × 247 컬럼)

**구성**
- 4D structure descriptor (Water/Hexane 3D SASA·NPSA·PSA, RMSD_All/BackBone, Desolvation_Free_Energy)
- Assay 의 RDKit 2D descriptor 200+ (MolWt, MolLogP, TPSA, qed, BCUT2D, Chi*, Kappa*, PEOE/SMR/SlogP/EState_VSA, fr_* 등)
- Target : `PAMPA` / `PAMPA-4D` (둘이 거의 동일, 회귀 target 으로 PAMPA-4D 사용)
- 보조 target : `Permeable = (PAMPA ≥ -6)` 분류용

**분석 흐름**
1. 데이터 품질 / 분포 / 메타 (Source, Length, Shape) 점검
2. PAMPA 분포 + 그룹별 차이
3. Feature ↔ PAMPA 상관 / 다중공선성
4. PCA 저차원 시각화
5. **회귀 모델 비교 — RandomForest / Lasso (L1) / Ridge (L2) / ElasticNet**
6. 모델 해석 — coefficient (선형) / feature importance (RF) / permutation importance
7. 분류 모델 비교 — RandomForest / LogReg-L1 / LogReg-L2
8. 종합 요약

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.unicode_minus'] = False

PATH = '/ssd0/sohyun/cyclic_peptide_permeability/data/CycPeptMPDB-4D_with_assay_descriptors.csv'
df = pd.read_csv(PATH)

# 분류 보조 라벨
PERM_THRESHOLD = -6.0
df['Permeable'] = (df['PAMPA'] >= PERM_THRESHOLD).astype(int)

print(f'shape : {df.shape}')
print(f'Permeable ratio : {df["Permeable"].mean():.3%}')
df.head()

## 1. 데이터 품질 점검

In [ ]:
miss = df.isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
print(f'결측 있는 컬럼 ({len(miss)}개):')
print(miss)

print(f'\n전체 row : {len(df)}')
print(f'CycPeptMPDB_ID 중복 : {df["CycPeptMPDB_ID"].duplicated().sum()}')
print(f'완전 동일 row 중복   : {df.duplicated().sum()}')

In [ ]:
# 모델링 가능한 numeric feature 추리기
exclude_keys = ('Caco2','MDCK','RRCK','T_PAMPA','Detection','R_PAMAP','R_Caco2',
                 'PAMPA','Permeability','Permeable','HELM','Source','Sequence','SMILES',
                 'Original_Name','CycPeptMPDB_ID','Year','Version','Structurally_Unique_ID',
                 'Molecule_Shape','Ipc','NULL')

numeric_feats = []
for c in df.columns:
    if any(k in c for k in exclude_keys):
        continue
    s = pd.to_numeric(df[c], errors='coerce')
    miss = s.isna().mean()
    if miss < 0.01 and s.nunique(dropna=True) > 5 \
       and np.isfinite(s.dropna()).all() and s.abs().max() < 1e30:
        numeric_feats.append(c)

# 4D vs Assay 라벨
is_4d = lambda c: any(k in c for k in ['Water_3D','Hexane_3D','avgRMSD','Desolvation_Free_Energy']) or c in ['Monomer_Length','Monomer_Length_in_Main_Chain']
feat_4d = [c for c in numeric_feats if is_4d(c)]
feat_as = [c for c in numeric_feats if not is_4d(c)]
print(f'사용 가능한 numeric feature : {len(numeric_feats)}')
print(f'  - 4D-side                : {len(feat_4d)}')
print(f'  - Assay-side             : {len(feat_as)}')
print('\n4D feature:')
print(feat_4d)
print('\nAssay feature 처음 30개:')
print(feat_as[:30])

## 2. PAMPA 분포 & 그룹별 차이

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['PAMPA'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(PERM_THRESHOLD, color='red', ls='--', label=f'threshold = {PERM_THRESHOLD}')
axes[0].set(xlabel='PAMPA', ylabel='count', title='PAMPA distribution')
axes[0].legend()

labels = ['Non-perm\n(<-6)', 'Permeable\n(>=-6)']
counts = [(df['Permeable']==0).sum(), (df['Permeable']==1).sum()]
axes[1].bar(labels, counts, color=['#d9534f','#5cb85c'])
for i, v in enumerate(counts):
    axes[1].text(i, v, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom')
axes[1].set(title='Class balance', ylabel='count')
plt.tight_layout(); plt.show()

print(f'PAMPA  mean={df["PAMPA"].mean():.3f}  std={df["PAMPA"].std():.3f}  '
       f'skew={df["PAMPA"].skew():.3f}  kurt={df["PAMPA"].kurt():.3f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Source
src_order = df['Source'].value_counts().index
sns.boxplot(data=df, x='Source', y='PAMPA', order=src_order, ax=axes[0,0], palette='Set2')
axes[0,0].axhline(-6, color='red', ls='--', alpha=0.5)
axes[0,0].tick_params(axis='x', rotation=20)
axes[0,0].set_title('PAMPA by Source')

# Permeable rate per Source
rate = df.groupby('Source')['Permeable'].agg(['mean','count']).reindex(src_order)
axes[0,1].bar(rate.index, rate['mean'], color='teal')
for i, (m, n) in enumerate(zip(rate['mean'], rate['count'])):
    axes[0,1].text(i, m, f'{m:.1%}\n(n={n})', ha='center', va='bottom', fontsize=9)
axes[0,1].tick_params(axis='x', rotation=20)
axes[0,1].set(ylabel='permeable rate', ylim=(0,1.1), title='Permeable rate by Source')

# Length
sns.boxplot(data=df, x='Monomer_Length', y='PAMPA', ax=axes[1,0], palette='viridis')
axes[1,0].axhline(-6, color='red', ls='--', alpha=0.5)
axes[1,0].set_title('PAMPA by Monomer_Length')

# Shape
sns.violinplot(data=df, x='Molecule_Shape', y='PAMPA', ax=axes[1,1], palette='pastel', inner='quartile')
axes[1,1].axhline(-6, color='red', ls='--', alpha=0.5)
axes[1,1].set_title('PAMPA by Molecule_Shape')
plt.tight_layout(); plt.show()

## 3. Feature ↔ PAMPA 상관

In [ ]:
rows = []
for c in numeric_feats:
    sub = df[[c,'PAMPA']].dropna()
    if len(sub) < 100: continue
    r, p = stats.pearsonr(sub[c], sub['PAMPA'])
    rs, ps = stats.spearmanr(sub[c], sub['PAMPA'])
    rows.append({'feature': c, 'pearson_r': r, 'spearman_r': rs,
                  'side': '4D' if c in feat_4d else 'Assay'})
corr_df = pd.DataFrame(rows).sort_values('pearson_r', key=abs, ascending=False)

# Top 25
print('|Pearson r| 상위 25 feature:')
display(corr_df.head(25).round(4))

fig, ax = plt.subplots(figsize=(10, 8))
top = corr_df.head(25).iloc[::-1]
colors = ['steelblue' if s == '4D' else 'orange' for s in top['side']]
ax.barh(top['feature'], top['pearson_r'], color=colors)
import matplotlib.patches as mpatches
ax.legend(handles=[mpatches.Patch(color='steelblue', label='4D'),
                    mpatches.Patch(color='orange', label='Assay')])
ax.axvline(0, color='k', lw=0.5)
ax.set(title='Top 25 features by |Pearson r| with PAMPA', xlabel='Pearson r')
plt.tight_layout(); plt.show()

In [ ]:
# 상위 15 feature 간 상관행렬 — 다중공선성 체크
top15 = corr_df.head(15)['feature'].tolist()
cm = df[top15 + ['PAMPA']].apply(pd.to_numeric, errors='coerce').corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(cm, dtype=bool), k=1)
sns.heatmap(cm, mask=mask, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
             annot_kws={'size':7}, vmin=-1, vmax=1, ax=ax)
ax.set_title('Top 15 features + PAMPA — correlation matrix')
plt.tight_layout(); plt.show()

## 4. PCA 저차원 시각화

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df[numeric_feats].apply(pd.to_numeric, errors='coerce')
X = X.replace([np.inf, -np.inf], np.nan).dropna()
y_pampa = df.loc[X.index, 'PAMPA']
y_perm  = df.loc[X.index, 'Permeable']
shape   = df.loc[X.index, 'Molecule_Shape']

Xs = StandardScaler().fit_transform(X)
pca = PCA(n_components=6, random_state=42).fit(Xs)
Z = pca.transform(Xs)

print('Explained variance ratio :', pca.explained_variance_ratio_.round(3))
print('Cumulative               :', np.cumsum(pca.explained_variance_ratio_).round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sc = axes[0].scatter(Z[:,0], Z[:,1], c=y_pampa, cmap='RdYlGn', s=8, alpha=0.6)
axes[0].set(title='PC1 vs PC2 (PAMPA)', xlabel='PC1', ylabel='PC2')
plt.colorbar(sc, ax=axes[0], label='PAMPA')

for lbl, c in zip([0, 1], ['#d9534f','#5cb85c']):
    m = y_perm == lbl
    axes[1].scatter(Z[m, 0], Z[m, 1], c=c, s=8, alpha=0.5,
                     label=('Perm' if lbl else 'Non-perm'))
axes[1].set(title='PC1 vs PC2 (permeability)', xlabel='PC1', ylabel='PC2'); axes[1].legend()

for s, c in zip(shape.unique(), ['#5bc0de','#f0ad4e']):
    m = shape == s
    axes[2].scatter(Z[m, 0], Z[m, 1], c=c, s=8, alpha=0.5, label=s)
axes[2].set(title='PC1 vs PC2 (shape)', xlabel='PC1', ylabel='PC2'); axes[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
loadings = pd.DataFrame(pca.components_[:4].T,
                         index=numeric_feats,
                         columns=[f'PC{i+1}' for i in range(4)])
# PC1 절댓값 큰 top 15
top_pc1 = loadings.iloc[loadings['PC1'].abs().argsort()[::-1].values[:15]]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(top_pc1, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('PCA loadings — top 15 features by |PC1|')
plt.tight_layout(); plt.show()

## 5. 회귀 모델 비교 — RandomForest / Lasso(L1) / Ridge(L2) / ElasticNet

Target : `PAMPA-4D` (= PAMPA, 동일 값). Train/Test 80/20 stratified by Permeable, 5-fold CV (train 내) 로 일반화 성능 비교.

- 선형 모델 (Lasso/Ridge/EN) : `StandardScaler` 적용 + 내부 CV 로 α 자동 선택
- RandomForest : 원본 스케일

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 모델 입력 만들기
X = df[numeric_feats].apply(pd.to_numeric, errors='coerce')
X = X.replace([np.inf, -np.inf], np.nan)
valid = X.dropna().index
X = X.loc[valid].values
y = df.loc[valid, 'PAMPA'].values
perm_lbl = df.loc[valid, 'Permeable'].values
feat_names = np.array(numeric_feats)

Xtr, Xte, ytr, yte, ptr, pte = train_test_split(
    X, y, perm_lbl, test_size=0.2, random_state=42, stratify=perm_lbl)
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

print(f'Train : {Xtr.shape},  Test : {Xte.shape}')
print(f'Permeable ratio (train/test): {ptr.mean():.3f} / {pte.mean():.3f}')

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

reg_models = {
    'Lasso(L1)':  (LassoCV(alphas=np.logspace(-4, 1, 30), cv=5,
                            random_state=42, max_iter=50000, tol=1e-3), True),
    'Ridge(L2)':  (RidgeCV(alphas=np.logspace(-3, 3, 30)), True),
    'ElasticNet': (ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                                 alphas=np.logspace(-4, 1, 30),
                                 cv=5, random_state=42, max_iter=50000, tol=1e-3), True),
    'RandomForest': (RandomForestRegressor(
                       n_estimators=400, max_depth=None,
                       n_jobs=-1, random_state=42), False),
}

results, preds, fitted = [], {}, {}
for name, (mdl, scaled) in reg_models.items():
    Xa, Xb = (Xtr_s, Xte_s) if scaled else (Xtr, Xte)
    mdl.fit(Xa, ytr)
    p = mdl.predict(Xb)
    cv = cross_val_score(mdl, Xa, ytr, cv=kf, scoring='r2', n_jobs=-1)
    results.append({
        'model':       name,
        'cv_R2_mean':  cv.mean(),
        'cv_R2_std':   cv.std(),
        'train_R2':    r2_score(ytr, mdl.predict(Xa)),
        'test_R2':     r2_score(yte, p),
        'test_RMSE':   float(np.sqrt(mean_squared_error(yte, p))),
        'test_MAE':    mean_absolute_error(yte, p),
    })
    preds[name] = p
    fitted[name] = mdl

reg_df = pd.DataFrame(results).sort_values('test_R2', ascending=False).reset_index(drop=True)
reg_df.round(4)

In [ ]:
n = len(preds)
fig, axes = plt.subplots(1, n, figsize=(4.2*n, 4.2))
if n == 1: axes = [axes]
for ax, (name, p) in zip(axes, preds.items()):
    ax.scatter(yte, p, s=10, alpha=0.35)
    ax.plot([yte.min(), yte.max()], [yte.min(), yte.max()], 'r--')
    ax.set(xlabel='Actual PAMPA', ylabel='Predicted PAMPA',
            title=f'{name}  R²={r2_score(yte, p):.3f}')
plt.tight_layout(); plt.show()

# CV vs Test R² 비교
plot_df = reg_df.set_index('model')
x = np.arange(len(plot_df))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x-0.2, plot_df['cv_R2_mean'], yerr=plot_df['cv_R2_std'], width=0.4, label='CV R² (train)')
ax.bar(x+0.2, plot_df['test_R2'], width=0.4, label='Test R²', color='orange')
ax.set_xticks(x); ax.set_xticklabels(plot_df.index, rotation=15)
ax.axhline(0, color='k', lw=0.5)
ax.set(ylabel='R²', title='Regression R² comparison')
ax.legend(); plt.tight_layout(); plt.show()

## 6. 모델 해석 — coefficient / importance

In [ ]:
# Lasso / Ridge / ElasticNet coefficient
lasso = fitted['Lasso(L1)']; ridge = fitted['Ridge(L2)']; enet = fitted['ElasticNet']
lasso_coef = pd.Series(lasso.coef_, index=feat_names)
ridge_coef = pd.Series(ridge.coef_, index=feat_names)
enet_coef  = pd.Series(enet.coef_,  index=feat_names)

print(f'Lasso α      : {lasso.alpha_:.5f},   non-zero = {(lasso_coef != 0).sum()} / {len(feat_names)}')
print(f'Ridge α      : {ridge.alpha_:.5f}')
print(f'ElasticNet α : {enet.alpha_:.5f}, l1_ratio = {enet.l1_ratio_:.2f}')

top_k = 25
union_mag = pd.concat([lasso_coef.abs(), ridge_coef.abs(), enet_coef.abs()], axis=1).max(axis=1)
top_feats = union_mag.nlargest(top_k).index

comp = pd.DataFrame({
    'Lasso(L1)':  lasso_coef.reindex(top_feats),
    'Ridge(L2)':  ridge_coef.reindex(top_feats),
    'ElasticNet': enet_coef.reindex(top_feats),
})
fig, ax = plt.subplots(figsize=(11, 9))
comp.iloc[::-1].plot(kind='barh', ax=ax, width=0.8)
ax.axvline(0, color='k', lw=0.6)
ax.set(title=f'Linear coefficients (top {top_k} by |coef|)', xlabel='coef (scaled X)')
plt.tight_layout(); plt.show()
comp.round(4)

In [ ]:
# Lasso 가 살린 feature
kept = lasso_coef[lasso_coef != 0].sort_values(key=abs, ascending=False)
kept_df = kept.to_frame('lasso_coef')
kept_df['side'] = ['4D' if c in feat_4d else 'Assay' for c in kept_df.index]
print(f'Lasso 가 유지한 feature ({len(kept)}개) — 상위 20:')
display(kept_df.head(20).round(5))
print(f'\n4D 측 유지: {(kept_df["side"] == "4D").sum()} / {len(feat_4d)}')
print(f'Assay 측 유지: {(kept_df["side"] == "Assay").sum()} / {len(feat_as)}')

In [ ]:
# RandomForest feature importance
rf = fitted['RandomForest']
imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
imp_df = imp.head(25).to_frame('importance').reset_index().rename(columns={'index':'feature'})
imp_df['side'] = imp_df['feature'].apply(lambda c: '4D' if c in feat_4d else 'Assay')

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['steelblue' if s == '4D' else 'orange' for s in imp_df['side']]
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color=colors[::-1])
import matplotlib.patches as mpatches
ax.legend(handles=[mpatches.Patch(color='steelblue', label='4D'),
                    mpatches.Patch(color='orange', label='Assay')])
ax.set(title='RandomForest top-25 feature importance', xlabel='importance')
plt.tight_layout(); plt.show()

# 출처별 importance 합
tot_4d = imp[imp.index.isin(feat_4d)].sum()
tot_as = imp[imp.index.isin(feat_as)].sum()
print(f'전체 importance — 4D : {tot_4d:.3f} ({tot_4d*100:.1f}%)')
print(f'전체 importance — Assay: {tot_as:.3f} ({tot_as*100:.1f}%)')

In [ ]:
from sklearn.inspection import permutation_importance
perm = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=42, n_jobs=-1)
perm_df = pd.DataFrame({
    'feature': feat_names,
    'mean':    perm.importances_mean,
    'std':     perm.importances_std,
}).sort_values('mean', ascending=False)

K = 20
fig, ax = plt.subplots(figsize=(10, 7))
sub = perm_df.head(K).iloc[::-1]
colors = ['steelblue' if c in feat_4d else 'orange' for c in sub['feature']]
ax.barh(sub['feature'], sub['mean'], xerr=sub['std'], color=colors)
ax.set(title=f'Permutation importance — RF (top {K}, test set)',
        xlabel='mean decrease in R²')
plt.tight_layout(); plt.show()
perm_df.head(20).round(4)

## 7. 분류 모델 — RandomForest / LogReg-L1 / LogReg-L2

Target : `Permeable = (PAMPA ≥ −6)`, threshold 기반 이진 분류.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix, ConfusionMatrixDisplay

cls_models = {
    'LogReg-L1':  (LogisticRegressionCV(Cs=np.logspace(-3, 2, 20), cv=5,
                                          penalty='l1', solver='saga', max_iter=30000,
                                          n_jobs=-1, scoring='roc_auc', tol=1e-3), True),
    'LogReg-L2':  (LogisticRegressionCV(Cs=np.logspace(-3, 2, 20), cv=5,
                                          penalty='l2', solver='lbfgs', max_iter=30000,
                                          n_jobs=-1, scoring='roc_auc'), True),
    'RandomForest': (RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=42,
                                              class_weight='balanced'), False),
}

cls_results, cls_probs, cls_fitted = [], {}, {}
for name, (mdl, scaled) in cls_models.items():
    Xa, Xb = (Xtr_s, Xte_s) if scaled else (Xtr, Xte)
    mdl.fit(Xa, ptr)
    prob = mdl.predict_proba(Xb)[:, 1]
    pred = (prob >= 0.5).astype(int)
    cls_probs[name]  = prob
    cls_fitted[name] = mdl
    cls_results.append({
        'model':    name,
        'test_AUC': roc_auc_score(pte, prob),
        'test_Acc': float((pred == pte).mean()),
    })

cls_df = pd.DataFrame(cls_results).sort_values('test_AUC', ascending=False).reset_index(drop=True)
cls_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, prob in cls_probs.items():
    fpr, tpr, _ = roc_curve(pte, prob)
    auc = roc_auc_score(pte, prob)
    ax.plot(fpr, tpr, label=f'{name}  AUC={auc:.3f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set(xlabel='FPR', ylabel='TPR', title='ROC — Permeable classification')
ax.legend(); plt.tight_layout(); plt.show()

# Best model confusion matrix
best = cls_df.iloc[0]['model']
mdl  = cls_fitted[best]
Xb   = Xte_s if best.startswith('LogReg') else Xte
y_pred = mdl.predict(Xb)
print(f'>> Best : {best}')
print(classification_report(pte, y_pred, target_names=['Non-perm','Perm']))
ConfusionMatrixDisplay(confusion_matrix(pte, y_pred), display_labels=['Non-perm','Perm']).plot(cmap='Blues')
plt.title(f'{best} — confusion matrix'); plt.show()

In [ ]:
# LogReg-L1 / L2 coefficient 비교
l1 = cls_fitted['LogReg-L1']; l2 = cls_fitted['LogReg-L2']
l1_coef = pd.Series(l1.coef_.ravel(), index=feat_names)
l2_coef = pd.Series(l2.coef_.ravel(), index=feat_names)
print(f'LogReg-L1 best C = {float(l1.C_[0]):.4g}, non-zero = {(l1_coef != 0).sum()} / {len(feat_names)}')
print(f'LogReg-L2 best C = {float(l2.C_[0]):.4g}')

top_k = 20
top_feats = pd.concat([l1_coef.abs(), l2_coef.abs()], axis=1).max(axis=1).nlargest(top_k).index
comp = pd.DataFrame({'LogReg-L1': l1_coef.reindex(top_feats),
                      'LogReg-L2': l2_coef.reindex(top_feats)})
fig, ax = plt.subplots(figsize=(10, 7))
comp.iloc[::-1].plot(kind='barh', ax=ax, width=0.8)
ax.axvline(0, color='k', lw=0.6)
ax.set(title=f'Logistic regression coefficients (top {top_k})', xlabel='coef (scaled X)')
plt.tight_layout(); plt.show()

## 8. 종합 요약 (검증된 실제 수치)

### 데이터셋
- 5160 펩타이드 × 247 컬럼, **사용 가능한 numeric feature 114개** (4D 11 + Assay 103)
- 결측은 다른 어세이값 (`Caco2`, `MDCK`, `RRCK`, `T_PAMPA`) 만 있음 — 모델 입력엔 결측 없음
- Permeable rate 65.5%, **Source 5개**, Length {6, 7, 10}, Shape {Circle, Lariat}

### 회귀 성능 (PAMPA 예측, n_features = 114)
| 모델 | CV R² (5-fold) | Test R² | Test RMSE | Test MAE |
|---|---|---|---|---|
| **RandomForest** | **0.302 ± 0.022** | **0.373** | **0.797** | **0.497** |
| ElasticNet       | 0.223 ± 0.038      | 0.288  | 0.849     | 0.552    |
| Lasso (L1)       | 0.227 ± 0.028      | 0.287  | 0.850     | 0.553    |
| Ridge (L2)       | 0.192 ± 0.106      | 0.292  | 0.847     | 0.550    |

- **RandomForest 가 Test R² 기준 최상** — 0.37 vs 선형 모델 0.29 (∼28% 개선) → PAMPA-feature 관계가 비선형.
- 단 RF 의 train R² 0.90 vs CV 0.30 → 다소 overfit, 하지만 test 에서도 가장 좋은 성능 유지.
- **Lasso α = 3.3e-4** : 114 feature 중 **64개 (56%) 만 유지** (4D 9/11, Assay 55/103) → 절반 정도가 "필요한" feature.

### 분류 성능 (Permeable, threshold = −6)
| 모델 | Test AUC | Test Acc |
|---|---|---|
| **RandomForest** | **0.853** | **0.784** |
| LogReg-L2        | 0.824    | 0.766    |
| LogReg-L1        | 0.819    | 0.772    |

- RF best, LogReg-L2 와 -L1 차이는 미미 → 선형 분리가 어느 정도 가능하나 RF 의 비선형성이 추가 이득.
- best classification report: precision 0.81 / recall 0.88 / F1 0.84 (Permeable 클래스 기준).

### Feature 중요도 — 4D vs Assay
- **RandomForest 전체 importance** : 4D **44.0%** vs Assay **56.0%** — 4D 11개 feature 가 Assay 103개와 비슷한 비중을 차지 → **개별 4D feature 의 정보 밀도가 매우 높음**.
- **Lasso 가 유지한 feature** : 4D 9/11 (82%) vs Assay 55/103 (53%) — Lasso 가 4D feature 를 거의 전부 살림 → 4D 의 중복도가 낮음.
- **Linear coefficient top** : `PC1`, `Chi3n`, `Chi4n`, `SMR_VSA10`, `FpDensityMorgan2/3` 등 Assay 측 RDKit topological/VSA descriptor 가 강한 영향. 4D 측은 absolute coef 가 작아도 분산을 다른 차원에서 잡음 (RF importance 와 대조).

### 다음 단계
1. **GroupKFold by Source** : 4D 가 5 source 한정이므로 OOD 일반화를 정직하게 평가.
2. **PAMPA = −10 (detection limit) sentinel 처리** : censored data 로 Tobit / 2-stage 모델, 또는 분류만 학습.
3. **Redundant pair 제거** (`Water_3D_SASA` ≈ `LabuteASA`, `Water_3D_PSA` ≈ `TPSA`) → 차원 축소 + 해석성 개선.
4. **RF 의 overfit 완화** : `max_depth`, `min_samples_leaf` 튜닝 또는 XGBoost / LightGBM 비교.
5. **SHAP 분석** : RF 가 특정 펩타이드를 잘못 예측할 때 어떤 feature 가 잘못 작용했는지 개별 해석.